# 测试 ONNX Relax

创建缓存目录：

In [1]:
from pathlib import Path

temp_dir = Path(".temp")
temp_dir.mkdir(exist_ok=True)

## 构建 ONNX 模型

In [2]:
import torch
import torch.nn.functional as F
class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(3, 16, 3, bias=False)
        self.conv2 = torch.nn.Conv2d(16, 32, 1, bias=False)

    def forward(self, x):
        # x = self.conv(x)
        x = F.interpolate(x, size=None, scale_factor=(0.5, 0.5), mode="nearest",)
        return x


torch_model = M()
input_tensor = torch.randn(1, 3, 10, 10)
torch.onnx.export(
    torch_model, 
    (input_tensor,), 
    temp_dir/"test.onnx", 
    input_names=["x"],
    opset_version=11,
)
torch.onnx.export(
    torch_model, 
    (input_tensor,), 
    temp_dir/"test19.onnx", 
    input_names=["x"],
    opset_version=19,
)

## 转换 ONNX 模型为 Relax 模型

In [3]:
import onnx
from tvm.relax.frontend.onnx import from_onnx
model = onnx.load(temp_dir/"test.onnx")
tvm_model = from_onnx(model,  keep_params_in_input=True)

roi: metadata["relax.expr.Constant"][0]
# Metadata omitted. Use show_meta=True in script() method to show it.
scales: metadata["relax.expr.Constant"][0]
# Metadata omitted. Use show_meta=True in script() method to show it.
sizes: None
ndims: 4
mode: nearest_neighbor
sizes: [T.int64(5), T.int64(5)]
roi: R.concat((R.strided_slice(metadata["relax.expr.Constant"][0], (R.prim_value(0),), (R.prim_value(2),), (R.prim_value(4),), assume_inbound=False), R.strided_slice(metadata["relax.expr.Constant"][0], (R.prim_value(0),), (R.prim_value(6),), (R.prim_value(8),), assume_inbound=False)), axis=0)
# Metadata omitted. Use show_meta=True in script() method to show it.
x: x
Error converting operator Resize, with inputs: [x, metadata["relax.expr.Constant"][0]
# Metadata omitted. Use show_meta=True in script() method to show it., metadata["relax.expr.Constant"][0]
# Metadata omitted. Use show_meta=True in script() method to show it.]


TVMError: Traceback (most recent call last):
  File "/media/pc/data/lxw/ai/tvm/include/tvm/runtime/packed_func.h", line 924
TVMError: In function relax.op.image.resize2d(0: RelaxExpr, 1: RelaxExpr, 2: Array<FloatImm>, 3: runtime.String, 4: runtime.String, 5: runtime.String, 6: runtime.String, 7: double, 8: int, 9: double, 10: DataType) -> RelaxExpr: error while converting argument 2: [11:06:39] /media/pc/data/lxw/ai/tvm/include/tvm/runtime/packed_func.h:2274: InternalError: Check failed: (!checked_type.defined()) is false: Expected Array[runtime.Object], but got relax.expr.Call


In [4]:
import onnx
from tvm.relax.frontend.onnx import from_onnx
model = onnx.load(temp_dir/"test19.onnx")
tvm_model = from_onnx(model,  keep_params_in_input=True)

roi: None
scales: metadata["relax.expr.Constant"][0]
# Metadata omitted. Use show_meta=True in script() method to show it.
sizes: None
ndims: 4
mode: nearest_neighbor
sizes: [T.int64(5), T.int64(5)]
roi: [0.0, 0.0, 0.0, 0.0]
x: x


In [5]:
tvm_model

# from tvm.script import ir as I
# from tvm.script import tir as T
# from tvm.script import relax as R

@I.ir_module
class Module:
    @R.function
    def main(x: R.Tensor((1, 3, 10, 10), dtype="float32")) -> R.Tensor((1, 3, 5, 5), dtype="float32"):
        R.func_attr({"num_input": 1})
        with R.dataflow():
            gv: R.Tensor((1, 3, 5, 5), dtype="float32") = R.image.resize2d(x, R.shape([5, 5]), roi=[T.float32(0.0), T.float32(0.0), T.float32(0.0), T.float32(0.0)], layout="NCHW", method="nearest_neighbor", coordinate_transformation_mode="asymmetric", rounding_method="floor", cubic_alpha=-0.75, cubic_exclude=0, extrapolation_value=0.0, out_dtype="void")
            R.output(gv)
        return gv